# Dropbox shared-folder downloader

Stream every file from a Dropbox shared folder, resample to mono / 16 kHz, write it under `recordings/`, and delete the raw download right after so we never hold the un-resampled copy on disk.

Auth: `DROPBOX_KEY` in `.env` (treated as an OAuth2 access token).

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
from pathlib import Path
from urllib.parse import urlsplit, urlunsplit

import dropbox
import librosa
import pyrootutils
import soundfile as sf
from dotenv import load_dotenv
from dropbox.files import FileMetadata, SharedLink
from tqdm.auto import tqdm

ROOT = pyrootutils.setup_root(
    search_from=Path.cwd(),
    indicator="pyproject.toml",
    pythonpath=True,
    dotenv=True,
)
load_dotenv()

from building.utils.constants import SAMPLE_RATE

## Config

In [ ]:
SHARED_URL = "https://www.dropbox.com/scl/fo/5gk3iczix2jmlzoonvcwe/h/data/robert_lachlan/R21?e=2&subfolder_nav_tracking=1&dl=0"

OUT_DIR = ROOT / "recordings" / "robert_lachlan" / "R21"
TMP_DIR = ROOT / "recordings" / "_tmp"
OUT_DIR.mkdir(parents=True, exist_ok=True)
TMP_DIR.mkdir(parents=True, exist_ok=True)

CHUNK_BYTES = 4 * 1024 * 1024
AUDIO_EXTS = {".wav", ".flac", ".mp3", ".ogg", ".aif", ".aiff", ".m4a"}

print(f"target sample rate: {SAMPLE_RATE} Hz, mono")
print(f"output: {OUT_DIR}")

## Connect

In [ ]:
token = os.environ["DROPBOX_KEY"]
dbx = dropbox.Dropbox(token)
dbx.users_get_current_account().email

## Parse the shared URL

A Dropbox shared-folder URL looks like `https://www.dropbox.com/scl/fo/<id>/h/<subpath>?...`. The SDK wants the bare shared link (no `/h/...`, no query string); the subpath after `/h/` is then the `path` argument inside that shared root.

In [ ]:
def split_shared_url(url: str) -> tuple[str, str]:
    parts = urlsplit(url)
    segs = parts.path.split("/")
    try:
        h_idx = segs.index("h")
        root_path = "/".join(segs[:h_idx])
        subpath = "/" + "/".join(segs[h_idx + 1 :]).strip("/")
    except ValueError:
        root_path = parts.path
        subpath = ""
    root_url = urlunsplit((parts.scheme, parts.netloc, root_path, "", ""))
    return root_url, subpath.rstrip("/")


ROOT_URL, SUBPATH = split_shared_url(SHARED_URL)
print("root url:", ROOT_URL)
print("subpath :", repr(SUBPATH))

## List files

In [ ]:
def list_shared_folder(
    dbx: dropbox.Dropbox, root_url: str, subpath: str
) -> list[FileMetadata]:
    sl = SharedLink(url=root_url)
    res = dbx.files_list_folder(path=subpath, shared_link=sl, recursive=True)
    entries = list(res.entries)
    while res.has_more:
        res = dbx.files_list_folder_continue(res.cursor)
        entries.extend(res.entries)
    return [e for e in entries if isinstance(e, FileMetadata)]


all_files = list_shared_folder(dbx, ROOT_URL, SUBPATH)
audio_files = [
    f for f in all_files if Path(f.name).suffix.lower() in AUDIO_EXTS
]
print(f"{len(all_files)} total entries, {len(audio_files)} audio files")
for f in audio_files[:5]:
    print(f"  {f.path_display}  ({f.size/1e6:.1f} MB)")

## Stream + resample

For each file:
1. Stream-download to `recordings/_tmp/` (chunked, never read whole file into memory at once).
2. Load + resample to mono / `SAMPLE_RATE` with librosa (streamed off disk, not the whole raw bytes again).
3. Write the resampled file under `recordings/...`.
4. Delete the raw temp file immediately.

In [ ]:
def stream_download(
    dbx: dropbox.Dropbox, root_url: str, file_path: str, dest: Path
) -> None:
    _md, resp = dbx.sharing_get_shared_link_file(url=root_url, path=file_path)
    try:
        with open(dest, "wb") as out:
            for chunk in resp.iter_content(chunk_size=CHUNK_BYTES):
                if chunk:
                    out.write(chunk)
    finally:
        resp.close()


def process(file_md: FileMetadata) -> str:
    rel = Path(file_md.path_display).relative_to(SUBPATH)
    out_path = (OUT_DIR / rel).with_suffix(".wav")
    out_path.parent.mkdir(parents=True, exist_ok=True)
    if out_path.exists():
        return "skip"

    tmp = TMP_DIR / file_md.id.replace(":", "_") / file_md.name
    tmp.parent.mkdir(parents=True, exist_ok=True)
    try:
        stream_download(dbx, ROOT_URL, file_md.path_display, tmp)
        sig, _ = librosa.load(
            str(tmp), sr=SAMPLE_RATE, mono=True, res_type="kaiser_fast"
        )
        sf.write(str(out_path), sig, SAMPLE_RATE, subtype="PCM_16")
    finally:
        if tmp.exists():
            tmp.unlink()
        try:
            tmp.parent.rmdir()
        except OSError:
            pass
    return "ok"


stats = {"ok": 0, "skip": 0, "err": 0}
for f in tqdm(audio_files, desc="R21"):
    try:
        stats[process(f)] += 1
    except Exception as e:
        stats["err"] += 1
        print(f"  fail {f.path_display}: {type(e).__name__}: {e}")

print(stats)
try:
    TMP_DIR.rmdir()
except OSError:
    pass